# Feature Extraction

Data representation plays a critical role in the performance of many machine learning methods in machine learning. The data representation of network traffic often determines the effectiveness of these models as much as the model itself. The wide range of novel events that network operators need to detect (e.g., attacks, malware, new applications, changes in traffic demands) introduces the possibility for a broad range of possible models and data representations.

[NetML](https://pypi.org/project/netml/) is an open-source tool and end-to-end pipeline for anomaly detection in network traffic. This notebook walks through the use of that library.

First, let us load the library.

In [1]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data

import pandas as pd

## Specify a Packet Capture File

Create a pcap data structure for which we would like to extract features. You could do this based on the packet capture files that we have been using in previous hands assignments. Any packet capture file will suffice, however.

You can define the minumum number of packets that you want to include in each flow.

In [2]:
pcap = PCAP('data/log4j.pcap')
pcap.pcap2pandas()

pdf = pcap.df
pdf.head()

,datetime,dns_query,dns_resp,ip_dst,ip_dst_int,ip_src,ip_src_int,is_dns,length,mac_dst,mac_dst_int,mac_src,mac_src_int,port_dst,port_src,protocol,time,time_normed
0,2021-12-15 22:35:00,None,None,198.71.247.91,3326605147,128.14.134.170,2148435626,False,74,00:16:3c:f1:fd:6d,95511772525,64:9e:f3:be:db:66,110633856981862,80.0,57468.0,TCP,1639604100.237882,0.000000
1,2021-12-15 22:35:00,None,None,128.14.134.170,2148435626,198.71.247.91,3326605147,False,74,64:9e:f3:be:db:66,110633856981862,00:16:3c:f1:fd:6d,95511772525,57468.0,80.0,TCP,1639604100.237939,0.000057
2,2021-12-15 22:35:00,None,None,198.71.247.91,3326605147,128.14.134.170,2148435626,False,66,00:16:3c:f1:fd:6d,95511772525,64:9e:f3:be:db:66,110633856981862,80.0,57468.0,TCP,1639604100.249425,0.011543
3,2021-12-15 22:35:00,None,None,198.71.247.91,3326605147,128.14.134.170,2148435626,False,271,00:16:3c:f1:fd:6d,95511772525,64:9e:f3:be:db:66,110633856981862,80.0,57468.0,TCP,1639604100.249475,0.011593
4,2021-12-15 22:35:00,None,None,128.14.134.170,2148435626,198.71.247.91,3326605147,False,66,64:9e:f3:be:db:66,110633856981862,00:16:3c:f1:fd:6d,95511772525,57468.0,80.0,TCP,1639604100.249525,0.011643


## Convert the Packet Capture Into Flows

Find the function in `netml` that converts the pcap file into flows. Examing the resulting data structure. What does it contain?

In [5]:
MINIMUM_PACKETS = 2

pcap = PCAP('data/log4j.pcap', flow_ptks_thres=MINIMUM_PACKETS)

pcap.pcap2flows()

In [17]:
# Look at the first flow in detail
first_key, first_packets = pcap.flows[0]
print("Flow key:", first_key)
for pkt in first_packets:
    pkt.show()          # full Scapy packet breakdown
    print(pkt.time)      # timestamp
    print(len(pkt))       # packet size in bytes
    break

Flow key: ('128.14.134.170', '198.71.247.91', 57468, 80, 6)
###[ Ethernet ]### 
  dst       = 00:16:3c:f1:fd:6d
  src       = 64:9e:f3:be:db:66
  type      = IPv4
###[ IP ]### 
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 60
     id        = 35444
     flags     = 
     frag      = 0
     ttl       = 52
     proto     = tcp
     chksum    = 0x37ec
     src       = 128.14.134.170
     dst       = 198.71.247.91
     \options   \
###[ TCP ]### 
        sport     = 57468
        dport     = www_http
        seq       = 3705618145
        ack       = 0
        dataofs   = 10
        reserved  = 0
        flags     = S
        window    = 29200
        chksum    = 0x13ad
        urgptr    = 0
        options   = [('MSS', 1460), ('SAckOK', b''), ('Timestamp', (1287894165, 0)), ('NOP', None), ('WScale', 7)]

1639604100.237882
74


> For a each packet in the flow, we get ethernet destination / port, IP information, and TCP information. The flow header provides us with the source ip address, destination ip address, source port, destination port, and protocol - all in a five tuple and ordered respectively.

## Explore the Flows

How many flows are in your data structure?

In [18]:
print(f"Total flows: {len(pcap.flows)}")

Total flows: 4795


What other information does the flow data structure contain, for each flow?

> The flow data structure contains both per-packet information and the five-tuple as stated above. 

## Extract Features from Each Flow

Use the `netml` library to extract features from each flow. 

The [documentation](https://pypi.org/project/netml/) and [accompanying paper](https://arxiv.org/pdf/2006.16993.pdf) provide examples of features that you can try to extract. 

First try to extract the inter-arrival times for each flow.

### Interarrival Times

In [ ]:
# Extract inter-arrival time features
pcap.flow2features('IAT', fft=False, header=False)

iat_features = pcap.features

[[1.15430355e-02 5.00679016e-05 1.27089024e-02 5.04110408e+00
  9.73989964e-02]
 [1.15861893e-02 1.09100342e-03 5.00433397e+00 1.45769835e-01
  0.00000000e+00]
 [1.00105906e+00 2.02039385e+00 4.25303912e+00 0.00000000e+00
  0.00000000e+00]
 ...
 [5.70890903e-02 8.32796097e-04 5.00020003e+00 5.73501587e-02
  0.00000000e+00]
 [6.10411167e-02 3.00884247e-04 6.15091324e-02 5.00151205e+00
  0.00000000e+00]
 [6.13319874e-02 4.95910645e-04 5.00139904e+00 6.11410141e-02
  0.00000000e+00]]


### Explore the Per-Flow Features

Inspect and print the features for each flow. (If you feel compelled: Get fancy! Plot distributions, etc. Whatever you like!)

In [27]:
pcap.flow2features('SAMP_NUM', fft=False, header=False)

no_series = pcap.features

pcap.flow2features('SAMP_SIZE', fft=False, header=False)

size_series = pcap.features

print(no_series)
print(size_series)

[[4. 0. 0. 0. 0.]
 [3. 0. 0. 0. 0.]
 [1. 1. 0. 0. 0.]
 ...
 [3. 0. 0. 0. 0.]
 [4. 0. 0. 0. 0.]
 [3. 0. 0. 0. 0.]]
[[477.   0.   0.   0.   0.]
 [484.   0.   0.   0.   0.]
 [ 74.  74.   0.   0.   0.]
 ...
 [508.   0.   0.   0.   0.]
 [813.   0.   0.   0.   0.]
 [508.   0.   0.   0.   0.]]


### Other Features and Options

1. Try some of the other features in the `netml` library.

  Here are some of the other possibilities, which can be passed to the library:
  * IAT: A flow is represented as a timeseries of inter-arrival times between packets, i.e., elapsed time in seconds between any two packets in the flow.   
  *  STATS: A flow is represented as a set of statistical quantities. We choose ten of the most common such
statistics in the literature: flow duration, number of packets sent per second, number of bytes
per second, and various statistics on packet sizes within each flow: mean, standard deviation, inter-quartile range,
minimum, and maximum.
  * SIZE: A flow is represented as a timeseries of packet sizes in bytes, with one sample per packet. 
  * SAMP-NUM: A flow is partitioned into small intervals of equal length 𝛿𝑡, and the number of packets in each interval is recorded; thus, a flow is represented as a timeseries of packet counts in small time intervals, with one sample per time interval. Here, 𝛿𝑡 might be viewed as a choice of sampling rate for the timeseries, hence the nomenclature.
  * SAMP-SIZE: A flow is partitioned into time intervals of equal length 𝛿𝑡, and the total packet size (i.e., byte count) in each interval is recorded; thus, a flow is represented as a timeseries of byte counts in small time intervals, with one sample per time interval.
  

  
2. One of the challenges with providing packet traces to models involve ensuring that all feature vectors are of the same length. The `netml` libary will do that for you, but there are a number of different ways to solve the problem. What do some of the following options do?  Explore how different settings of the following affect the dimensionality of your resulting feature vector.

 * flow_ptks_thres
 * q_interval

## Thought Questions

What other features might you want to extract from packet captures that are not provided by the `netml` library?

> It could be good to easily have access to which packets are 'up' and which packets are 'down.'